# ByT5 BASELINE — Kaggle T4×2 (DataParallel)

Trains a vanilla ByT5 + classifier on the **full mixed train set** (all noise levels),
evaluates on all 4 test sets. This is the apples-to-apples baseline for the unified
NoiseBridge ByT5 run.

**Kaggle settings:**
- Accelerator: **GPU T4 × 2**
- Internet: ON
- Attach `hindimix-data` dataset

**Runtime estimate:** ~4–6h on T4×2

**Multi-seed:** run on 3 separate Kaggle accounts with SEED=42, 43, 44.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.40.0", "jellyfish", "sentencepiece", "accelerate"], check=True)

import torch
print('CUDA:', torch.cuda.is_available())
print('Num GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {name} ({mem:.1f} GB)')

In [ ]:
# ─── EDIT PER ACCOUNT ───
ENCODER = 'google/byt5-small'
SEED    = 42                      # 42 / 43 / 44

# ─── Hyperparameters (MUST MATCH NoiseBridge run for fair comparison) ───
EPOCHS      = 5
BATCH_SIZE  = 8
MAX_LEN     = 256
LR          = 3e-5
FP16        = True
NUM_WORKERS = 2

In [ ]:
import os

DATASET_NAME = 'hindimix-data'
DATA_DIR = f'/kaggle/input/{DATASET_NAME}'
if not os.path.exists(DATA_DIR):
    for d in os.listdir('/kaggle/input/'):
        if 'hindimix' in d.lower() or 'hinglish' in d.lower() or 'noisy' in d.lower():
            DATA_DIR = f'/kaggle/input/{d}'
            break

RESULTS_DIR = '/kaggle/working/results'
CKPT_DIR    = '/kaggle/working/checkpoints'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Data dir:', DATA_DIR)
print('Device :', DEVICE)
print('Files  :')
for f in sorted(os.listdir(DATA_DIR)):
    print(f'  {f}')

## ByT5 Baseline Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5EncoderModel


class ByT5Baseline(nn.Module):
    """Vanilla ByT5 encoder + mean-pool + classifier. Pure CE loss."""

    def __init__(self, encoder_name='google/byt5-small', num_labels=2, dropout=0.1):
        super().__init__()
        self.encoder     = T5EncoderModel.from_pretrained(encoder_name)
        self.hidden_size = self.encoder.config.d_model
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1).float()
        return (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    def predict(self, input_ids, attention_mask):
        return self.classifier(self.dropout(self.encode(input_ids, attention_mask)))

    def forward(self, input_ids, attention_mask, labels, class_weights=None):
        logits = self.predict(input_ids, attention_mask)
        loss = F.cross_entropy(logits, labels, weight=class_weights).unsqueeze(0)
        return {'loss': loss, 'logits': logits}

print('ByT5 baseline model defined.')

## Dataset + Training Utilities

In [ ]:
import os, json, random
import numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    ws = (torch.initial_seed() + worker_id) % (2**32)
    random.seed(ws); np.random.seed(ws)


class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.enc = tokenizer(texts, max_length=max_len, padding='max_length',
                             truncation=True, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {'input_ids': self.enc['input_ids'][idx],
                'attention_mask': self.enc['attention_mask'][idx],
                'labels': self.labels[idx]}


def collate(batch): return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


def load_data():
    """Unified: train on ALL rows of train.csv (clean + all noise levels).
       Eval on all 4 test sets (clean, low, medium, high)."""
    train_df = pd.read_csv(f'{DATA_DIR}/train.csv').dropna(subset=['text', 'label'])
    val_df   = pd.read_csv(f'{DATA_DIR}/val.csv').dropna(subset=['text', 'label'])
    test_dfs = {}
    for c in ['clean', 'low', 'medium', 'high']:
        p = f'{DATA_DIR}/test_clean.csv' if c == 'clean' else f'{DATA_DIR}/test_noisy_{c}.csv'
        if os.path.exists(p):
            test_dfs[c] = pd.read_csv(p).dropna(subset=['text', 'label'])
    return train_df, val_df, test_dfs


def unwrap(model):
    return model.module if isinstance(model, torch.nn.DataParallel) else model


def train_epoch(model, loader, optimizer, scheduler, class_weights, scaler=None):
    model.train()
    total = 0.0
    for batch in tqdm(loader, desc='  train', leave=False):
        optimizer.zero_grad()
        kwargs = {k: v.to(DEVICE) for k, v in batch.items()}
        kwargs['class_weights'] = class_weights

        if scaler is not None:
            with autocast(device_type='cuda'):
                out = model(**kwargs)
                loss = out['loss'].mean()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            out = model(**kwargs)
            loss = out['loss'].mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        total += loss.item()
    return total / max(len(loader), 1)


def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  eval', leave=False):
            logits = unwrap(model).predict(
                batch['input_ids'].to(DEVICE),
                batch['attention_mask'].to(DEVICE),
            )
            preds.extend(logits.argmax(dim=-1).cpu().tolist())
            targets.extend(batch['labels'].tolist())
    return f1_score(targets, preds, average='macro'), preds, targets

print('Utilities defined.')

## Train Baseline — click Save Version here to commit

In [ ]:
set_seed(SEED)

N_GPUS = torch.cuda.device_count()
print(f'{"="*72}')
print(f'ByT5 BASELINE | seed={SEED} | GPUs: {N_GPUS}')
print(f'fp16={FP16} | batch_size={BATCH_SIZE}')
print(f'Mode: UNIFIED (train once on mixed, eval on all 4 test sets)')
print(f'{"="*72}')

train_df, val_df, test_dfs = load_data()
tokenizer = AutoTokenizer.from_pretrained(ENCODER)

print(f'  Train: {len(train_df):,} rows')
print(f'  Val:   {len(val_df):,} rows')
print(f'  Test:  {list(test_dfs.keys())}')

train_ds = TextDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer, max_len=MAX_LEN)
val_ds   = TextDataset(val_df['text'].tolist(),   val_df['label'].tolist(),   tokenizer, max_len=MAX_LEN)
test_loaders = {
    c: DataLoader(TextDataset(tdf['text'].tolist(), tdf['label'].tolist(), tokenizer, max_len=MAX_LEN),
                  batch_size=BATCH_SIZE*2, collate_fn=collate,
                  num_workers=NUM_WORKERS, worker_init_fn=seed_worker)
    for c, tdf in test_dfs.items()
}

cw = compute_class_weight('balanced', classes=np.array([0,1]), y=train_df['label'].values)
class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)
print(f'  Class weights: [{cw[0]:.3f}, {cw[1]:.3f}]')

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, num_workers=NUM_WORKERS,
                          pin_memory=True, worker_init_fn=seed_worker, generator=g)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, collate_fn=collate,
                        num_workers=NUM_WORKERS, worker_init_fn=seed_worker)

model = ByT5Baseline(encoder_name=ENCODER).to(DEVICE)
if hasattr(model.encoder, 'gradient_checkpointing_enable'):
    model.encoder.gradient_checkpointing_enable()
    print('  Gradient checkpointing enabled.')
print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

if N_GPUS >= 2:
    model = torch.nn.DataParallel(model)
    print(f'  DataParallel: using {N_GPUS} GPUs')

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)
scaler = GradScaler('cuda') if FP16 else None

short = ENCODER.split('/')[-1].replace('-', '_')
run_name = f'baseline_{short}_all_s{SEED}'
ckpt_path = f'{CKPT_DIR}/{run_name}.pt'
best_val_f1 = 0.0

for epoch in range(EPOCHS):
    loss = train_epoch(model, train_loader, optimizer, scheduler, class_weights, scaler)
    val_f1, _, _ = evaluate(model, val_loader)
    print(f'  Epoch {epoch+1}/{EPOCHS} | loss:{loss:.4f} | val F1:{val_f1:.4f}')
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(unwrap(model).state_dict(), ckpt_path)
        print(f'    -> best saved (val F1: {best_val_f1:.4f})')

print(f'\n{"="*72}\nEVALUATING best checkpoint on all test sets\n{"="*72}')
unwrap(model).load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
all_results = []
for cond, tloader in test_loaders.items():
    test_f1, tp, tt = evaluate(model, tloader)
    print(f'\n[{cond}] Test F1 (macro): {test_f1:.4f}')
    print(classification_report(tt, tp, target_names=['Non-hate','Hate']))
    result = {
        'model': run_name, 'encoder': ENCODER, 'train_noise': 'all', 'test_noise': cond,
        'seed': SEED, 'test_f1_macro': round(test_f1, 4),
        'best_val_f1': round(best_val_f1, 4),
        'epochs': EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE, 'max_len': MAX_LEN,
        'num_gpus': N_GPUS,
    }
    with open(f'{RESULTS_DIR}/{run_name}_eval_{cond}.json', 'w') as f:
        json.dump(result, f, indent=2)
    all_results.append(result)

print('\nDone.')

## Summary + Download

In [ ]:
print('\n--- Results ---')
print(f'{"test":<10} {"seed":>5} {"val F1":>9} {"test F1":>9}')
for r in all_results:
    print(f'  {r["test_noise"]:<8} {r["seed"]:>5} {r["best_val_f1"]:>9.4f} {r["test_f1_macro"]:>9.4f}')

import shutil
shutil.make_archive('/kaggle/working/baseline_byt5_results', 'zip', RESULTS_DIR)
print('\nZipped -> /kaggle/working/baseline_byt5_results.zip')
print('Download from Output tab ->')